In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('rogii-wellbore-geology-prediction')

print("Path to competition files:", path)

In [ ]:
TRAIN_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/train"
TEST_DIR = "/kaggle/input/competitions/rogii-wellbore-geology-prediction/test"

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from scipy.interpolate import interp1d
from scipy.spatial import cKDTree
from concurrent.futures import ProcessPoolExecutor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

FEATURES = [
    'MD', 'Z', 'delta_Z', 'delta_MD', 'delta_X', 'delta_Y', 'dip_proxy', 'dip_acceleration',
    'GR', 'GR_rolling_mean_5', 'GR_rolling_mean_10', 'GR_rolling_mean_20',
    'GR_rolling_min_20', 'GR_rolling_max_20',
    'baseline_tvt_all_slope', 'baseline_tvt_recent_slope',
    'Distance_From_Known_Z', 'Spatial_Neighbor_Dip',
    'tort_ac_25', 'tort_ac_75', 'dls_roll', 'tort_ac_25_slope',
]

#Locally tunned with optuna for best params finding
BEST_LGB = {'learning_rate': 0.01046415691195076, 'num_leaves': 20, 'min_data_in_leaf': 177, 'feature_fraction': 0.5916511351553999,
            'bagging_fraction': 0.7964181538086376, 'lambda_l1': 0.0030028610554896184, 'lambda_l2': 2.7304292396750944, 'objective': 'regression',
            'metric': 'rmse', 'device': 'cpu', 'n_jobs': -1, 'random_state': 42, 'feature_pre_filter': False, 'verbose': -1}

BEST_XGB = {'learning_rate': 0.016087839839269442, 'max_depth': 5, 'min_child_weight': 89, 'subsample': 0.7490205090590788, 'colsample_bytree': 0.6483053115180675,
            'reg_alpha': 2.272806466182952, 'reg_lambda': 0.08212383345723068, 'objective': 'reg:squarederror', 'tree_method': 'hist', 'device': 'cuda', 'random_state': 42}

BEST_CAT = {'learning_rate': 0.09346567297070014, 'depth': 8, 'l2_leaf_reg': 0.13951085894901633, 'bagging_temperature': 0.032821359393250635,
            'iterations': 1000, 'task_type': 'GPU', 'random_seed': 42, 'verbose': False}


def compute_tortuosity(df, windows=(25, 75), span=10, dls_win=31):
    """Tortuosidad 3D: arc-to-chord (wiggle) + dogleg severity (steering).
    Alta tortuosidad => steering activo => formacion desviandose del plan."""
    n = len(df)
    out = {}
    if n < 5 or not all(c in df.columns for c in ['X','Y','Z','MD']):
        for W in windows: out[f'tort_ac_{W}'] = np.zeros(n)
        out['dls_roll'] = np.zeros(n); out['tort_ac_25_slope'] = np.zeros(n)
        return out
    P = np.column_stack([df['X'].values, df['Y'].values, df['Z'].values]).astype(float)
    idx = np.arange(n)
    seg = np.diff(P, axis=0)
    seglen = np.linalg.norm(seg, axis=1)
    cum = np.concatenate([[0.0], np.cumsum(seglen)])          # path acumulado, len n

    # 1) Arc-to-chord excess sobre ventanas trailing (wiggle a dos escalas)
    for W in windows:
        lo = np.clip(idx - W, 0, None)
        arc   = cum[idx] - cum[lo]
        chord = np.linalg.norm(P[idx] - P[lo], axis=1)
        out[f'tort_ac_{W}'] = np.clip(arc/(chord+1e-6) - 1.0, 0, None)   # 0 = recto

    # 2) Dogleg severity 3D (curvatura por ft), via direcciones suavizadas + arctan2
    hi = np.clip(idx + span, 0, n-1); lo2 = np.clip(idx - span, 0, n-1)
    D  = P[hi] - P[lo2]
    d0, d1 = D[:-1], D[1:]
    dot = np.sum(d0*d1, axis=1)
    crs = np.linalg.norm(np.cross(d0, d1), axis=1)
    ang = np.concatenate([[0.0], np.arctan2(crs, dot)])       # radianes, estable
    dmd = np.gradient(df['MD'].values.astype(float))
    dls = ang / (np.abs(dmd) + 1e-6)
    out['dls_roll'] = pd.Series(dls).rolling(dls_win, center=True, min_periods=1).mean().values

    # 3) Tendencia de la tortuosidad (¿el steering esta aumentando?)
    out['tort_ac_25_slope'] = pd.Series(out['tort_ac_25']).diff().rolling(
        dls_win, center=True, min_periods=1).mean().fillna(0).values
    return out


def run_particle_filter(hw, tw, n_particles=500, seed=42):
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0
    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input']); last_Z = float(last['Z']); last_MD = float(last['MD'])
    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))
    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values); dz = np.diff(tail['Z'].values); dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0
    N = n_particles; rng = np.random.default_rng(seed)
    ls   = last_tvt + last_Z
    pos  = ls + 2.0 * rng.standard_normal(N)        # init_spread = 2.0 ft
    rate = ir + 0.01 * rng.standard_normal(N)
    w    = np.ones(N) / N
    MOM=0.998; VN=0.002; PN=0.005; RP=0.1; RR=0.001; RESAMP=0.5
    md_v = ev['MD'].values.astype(float); z_v = ev['Z'].values.astype(float)
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())  # critico
    gr_v = gr_interp.values.astype(float)[ev.index]
    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev)); prev_MD = last_MD; log_lik = 0.0
    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - z_v[i], tw_tvt[0]-100, tw_tvt[-1]+100)
        pos   = tvt_p + z_v[i]
        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.maximum(np.exp(-0.5*np.minimum(d**2, 600.)), 1e-300)
        log_lik += np.log(max(float((w*lk).sum()), 1e-300))
        w = w * lk; ws = w.sum(); w = w/ws if ws>0 else np.ones(N)/N
        if 1.0/(w**2).sum() < RESAMP*N:
            cum = np.cumsum(w); u0 = rng.uniform(0, 1.0/N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N)/N), 0, N-1)
            pos = pos[idx] + RP*rng.standard_normal(N)
            rate = rate[idx] + RR*rng.standard_normal(N); w = np.ones(N)/N
        res[i] = float(np.dot(w, pos - z_v[i])); prev_MD = md_v[i]
    out_vals[list(ev.index)] = res
    return out_vals, log_lik

def build_spatial_map():
    print("--- 1. Building 3D Spatial Map ---")
    map_data = []
    for file in glob.glob(os.path.join(TRAIN_DIR, "*__horizontal_well.csv")):
        df_temp = pd.read_csv(file)
        valid_tvt = df_temp.dropna(subset=['TVT'])
        if len(valid_tvt) > 10:
            mean_x, mean_y = valid_tvt['X'].mean(), valid_tvt['Y'].mean()
            delta_tvt, delta_md = valid_tvt['TVT'].diff(), valid_tvt['MD'].diff()
            real_dip = (delta_tvt / delta_md).replace([np.inf, -np.inf], np.nan).mean()
            well_name = os.path.basename(file).split('__horizontal_well')[0]
            map_data.append([well_name, mean_x, mean_y, real_dip])

    df_spatial = pd.DataFrame(map_data, columns=['WELLNAME', 'X', 'Y', 'Real_Dip']).dropna()
    return df_spatial, cKDTree(df_spatial[['X', 'Y']].values)

def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=32, scale=5.0):
    preds, liks = [], []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles, seed=s)
        preds.append(p); liks.append(ll)
    liks = np.array(liks); wn = np.exp((liks - liks.max())/scale); wn /= wn.sum()
    return (wn[:,None] * np.stack(preds,0)).sum(0)

def process_single_well_pf(args):
    horiz_path, df_spatial_map, kdtree = args
    try:
        base_name = os.path.basename(horiz_path)
        well_name = base_name.split('__horizontal_well')[0]
        typewell_path = os.path.join(os.path.dirname(horiz_path), f"{well_name}__typewell.csv")
        if not os.path.exists(typewell_path): return None

        df_horiz = pd.read_csv(horiz_path)
        df_typewell = pd.read_csv(typewell_path)
        df_horiz['original_index'] = df_horiz.index
        df_horiz['WELLNAME'] = well_name
        df_horiz['is_blind_zone'] = df_horiz['TVT_input'].isna().astype(int)

        # Tortuosidad 3D
        tort = compute_tortuosity(df_horiz)
        for k, v in tort.items():
            df_horiz[k] = v

        # Geometry & Derivates
        df_horiz['delta_Z'] = df_horiz['Z'].diff().fillna(0)
        df_horiz['delta_MD'] = df_horiz['MD'].diff().fillna(0)
        df_horiz['delta_X'] = df_horiz['X'].diff().fillna(0)
        df_horiz['delta_Y'] = df_horiz['Y'].diff().fillna(0)
        df_horiz['dip_proxy'] = np.where(df_horiz['delta_MD'] != 0, df_horiz['delta_Z'] / df_horiz['delta_MD'], 0)
        df_horiz['dip_acceleration'] = df_horiz['dip_proxy'].diff().fillna(0)

        # Gamma Ray Signals
        for window in [5, 10, 20]:
            df_horiz[f'GR_rolling_mean_{window}'] = df_horiz['GR'].rolling(window=window, min_periods=1).mean()
            df_horiz[f'GR_rolling_min_{window}'] = df_horiz['GR'].rolling(window=window, min_periods=1).min().fillna(0)
            df_horiz[f'GR_rolling_max_{window}'] = df_horiz['GR'].rolling(window=window, min_periods=1).max().fillna(0)

        # Spatial Context
        current_x, current_y = df_horiz['X'].mean(), df_horiz['Y'].mean()
        _, indices = kdtree.query([[current_x, current_y]], k=4)
        df_horiz['Spatial_Neighbor_Dip'] = df_spatial_map.iloc[indices[0][1:]]['Real_Dip'].mean()

        # Flat Baseline Logic
        valid_data = df_horiz.dropna(subset=['TVT_input'])
        if not valid_data.empty:
            last_tvt, first_tvt = valid_data['TVT_input'].iloc[-1], valid_data['TVT_input'].iloc[0]
            last_z, last_md = valid_data['Z'].iloc[-1], valid_data['MD'].iloc[-1]
            first_md = valid_data['MD'].iloc[0]
            all_slope = (last_tvt - first_tvt) / (last_md - first_md) if (last_md - first_md) != 0 else 0
            if len(valid_data) >= 10:
                recent_tvt, recent_md = valid_data['TVT_input'].iloc[-10], valid_data['MD'].iloc[-10]
                recent_slope = (last_tvt - recent_tvt) / (last_md - recent_md) if (last_md - recent_md) != 0 else all_slope
            else:
                recent_slope = all_slope
        else:
            last_tvt = df_typewell['TVT'].median()
            last_z, all_slope, recent_slope = df_horiz['Z'].iloc[0], 0, 0

        df_horiz['last_known_TVT'] = last_tvt
        df_horiz['last_known_Z'] = last_z
        df_horiz['baseline_tvt_all_slope'] = all_slope
        df_horiz['baseline_tvt_recent_slope'] = recent_slope
        df_horiz['Distance_From_Known_Z'] = abs(df_horiz['Z'] - df_horiz['last_known_Z'])

        # === PARTICLE FILTER ===
        df_horiz['pf_residual'] = 0.0
        df_horiz['dip_pf'] = 0.0
        blind_mask = df_horiz['is_blind_zone'] == 1
        if blind_mask.sum() > 0:
            pf_pred = run_pf_lik_ensemble(df_horiz, df_typewell, n_particles=500, n_seeds=128, scale=5.0)
            blind_idx = df_horiz[blind_mask].index
            pf_blind_pred = pf_pred[blind_idx]

            df_horiz.loc[blind_idx, 'pf_residual'] = pf_blind_pred - last_tvt

            # dip instantáneo del camino del PF
            dip = np.diff(pf_blind_pred, prepend=pf_blind_pred[0] if len(pf_blind_pred) > 0 else 0)
            dmd = df_horiz.loc[blind_idx, 'delta_MD'].replace(0, np.nan).to_numpy()
            df_horiz.loc[blind_idx, 'dip_pf'] = np.nan_to_num(dip / dmd, nan=0.0, posinf=0.0, neginf=0.0)

            df_horiz['pf_pred'] = pf_pred

        if 'TVT' in df_horiz.columns:
            if 'pf_pred' in df_horiz.columns:
                 df_horiz['target_pf_residual'] = df_horiz['TVT'] - df_horiz['pf_pred']
            else:
                 df_horiz['target_pf_residual'] = df_horiz['TVT'] - df_horiz['last_known_TVT']

        return df_horiz
    except Exception as e:
        print(f"Error processing {horiz_path}: {e}")
        return None


def run_pipeline_pf():
    NEW_FEATURES = FEATURES.copy()
    # Cambiamos o agregamos las variables del PF
    NEW_FEATURES.extend(['pf_residual', 'dip_pf'])

    df_spatial_map, kdtree = build_spatial_map()

    print("\n--- 2. Processing Training Data with Particle Filter (Multicore) ---")
    train_files = glob.glob(os.path.join(TRAIN_DIR, "*__horizontal_well.csv"))
    args_train = [(f, df_spatial_map, kdtree) for f in train_files]
    with ProcessPoolExecutor() as exe:
        train_dfs = [res for res in tqdm(exe.map(process_single_well_pf, args_train), total=len(train_files)) if res is not None]

    df_train = pd.concat(train_dfs, ignore_index=True).dropna(subset=['target_pf_residual'])
    df_train_blind = df_train[df_train['is_blind_zone'] == 1].copy().reset_index(drop=True)

    X_train = df_train_blind[NEW_FEATURES]
    y_train = df_train_blind['target_pf_residual'] # Target is now the error of PF!
    groups = df_train_blind['WELLNAME']

    print("\n--- 3. Running 5-Fold Cross Validation (ML correcting PF) ---")
    gkf = GroupKFold(n_splits=5)
    oof_predictions = np.zeros(len(df_train_blind))

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
        print(f"  > Training Fold {fold + 1}/5...")
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va = X_train.iloc[val_idx]

        m_lgb = lgb.train(BEST_LGB, lgb.Dataset(X_tr, label=y_tr), num_boost_round=1000)
        m_xgb = xgb.train(BEST_XGB, xgb.DMatrix(X_tr, label=y_tr), num_boost_round=1000)
        m_cat = CatBoostRegressor(**BEST_CAT).fit(X_tr, y_tr)

        p_lgb = m_lgb.predict(X_va)
        p_xgb = m_xgb.predict(xgb.DMatrix(X_va))
        p_cat = m_cat.predict(X_va)
        oof_predictions[val_idx] = (p_lgb * 0.2146) + (p_xgb * 0.0114) + (p_cat * 0.7740)

    tvt_real = df_train_blind['TVT']
    # Recuperamos la prediccion total: PF + ml_correction
    final_oof_preds = df_train_blind['pf_pred'] + oof_predictions
    # guarda el OOF para poder re-diagnosticar sin recomputar el PF (que es lo caro)
    oof_dump = df_train_blind[['WELLNAME', 'MD', 'TVT', 'pf_pred']].copy()
    oof_dump['final_pred'] = final_oof_preds.values
    oof_dump.to_parquet('oof_dump.parquet')

    from diagnose_oof import diagnose_oof
    per_well, by_dist = diagnose_oof(df_train_blind, final_oof_preds.values)
    oof_rmse = np.sqrt(mean_squared_error(tvt_real, final_oof_preds))
    print(f"\nOUT-OF-FOLD (OOF) RMSE WITH PF + ENSEMBLE: {oof_rmse:.4f} feet\n")

    print("\n--- 4. Training Final Models on 100% Data ---")
    model_lgb = lgb.train(BEST_LGB, lgb.Dataset(X_train, label=y_train), num_boost_round=1000)
    model_xgb = xgb.train(BEST_XGB, xgb.DMatrix(X_train, label=y_train), num_boost_round=1000)
    model_cat = CatBoostRegressor(**BEST_CAT).fit(X_train, y_train)

    print("\n--- 5. Processing Test Data with Particle Filter ---")
    test_files = glob.glob(os.path.join(TEST_DIR, "*__horizontal_well.csv"))
    args_test = [(f, df_spatial_map, kdtree) for f in test_files]

    with ProcessPoolExecutor() as exe:
        test_dfs = [res[res['is_blind_zone'] == 1].copy() for res in tqdm(exe.map(process_single_well_pf, args_test), total=len(test_files)) if res is not None]

    df_test = pd.concat(test_dfs, ignore_index=True)

    p_test_lgb = model_lgb.predict(df_test[NEW_FEATURES])
    p_test_xgb = model_xgb.predict(xgb.DMatrix(df_test[NEW_FEATURES]))
    p_test_cat = model_cat.predict(df_test[NEW_FEATURES])

    ml_correction = (p_test_lgb * 0.40) + (p_test_xgb * 0.40) + (p_test_cat * 0.20)
    # Prediccion final = trayectoria PF + corrección del ML
    df_test['tvt_predicted'] = df_test['pf_pred'] + ml_correction

    submission = pd.DataFrame({
        'id': df_test['WELLNAME'] + '_' + df_test['original_index'].astype(str),
        'tvt': df_test['tvt_predicted']
    })
    submission['tvt'] = pd.to_numeric(submission['tvt'], errors='coerce').fillna(0)
    submission.to_csv('submission_pf_ensemble.csv', index=False)
    print("\nSaved submission_pf_ensemble.csv")
    display(submission.head())

if __name__ == '__main__':
    run_pipeline_pf()